# T07 — Residual, layer norm y feed-forward

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** el andamiaje sin el que la atención no entrena  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Ver por qué la conexión residual mantiene vivo el gradiente al apilar capas.
2. Comprobar el efecto normalizador de layer norm y el papel de la FFN por posición.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

La residual es una autopista por la que la información (y el gradiente) circula sin peajes: cada subcapa **añade** un ajuste en vez de reemplazar la señal. Layer norm evita que esa suma se descontrole capa tras capa.


## 5. Concepto mínimo

```text
salida = LayerNorm(x + Sublayer(x))
FFN(x) = max(0, x·W₁ + b₁)·W₂ + b₂      aplicada a CADA posición por separado
```

En el modelo base: `d_model = 512`, `d_ff = 2048`. La FFN es donde vive la mayor parte de los parámetros de cada bloque.


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
from ai_evolution.papers_lab import layer_norm

x = [3.0, -1.0, 0.5, 7.5]
n = layer_norm(x)
print('entrada   :', x)
print('layer norm:', [round(v, 4) for v in n])
print('media ≈', round(sum(n) / len(n), 6), '· varianza ≈', round(sum(v * v for v in n) / len(n), 4))

## 7. Predicción antes de ejecutar

Tras layer norm, ¿cuánto valdrán exactamente la media y la varianza del vector?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
señal = 1.0
print('SIN residual (cada capa multiplica por 0.8):')
s = señal
for capa in range(1, 13):
    s *= 0.8
    if capa % 4 == 0:
        print(f'  capa {capa:>2} → {s:.6f}')
print('CON residual (x + 0.8·x_ajuste, la identidad sobrevive):')
s = señal
for capa in range(1, 13):
    s = s + 0.8 * 0.1 * s
    if capa % 4 == 0:
        print(f'  capa {capa:>2} → {s:.6f}')

## 9. Salida interpretable

Sin residual la señal se apaga exponencialmente con la profundidad; con residual, el camino identidad la mantiene. Esto es lo que hace **apilables** los 6 bloques del paper (y los cientos de los modelos actuales).


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
import math
print('Layer norm sin epsilon, con un vector constante:')
v = [2.0, 2.0, 2.0]
media = sum(v) / len(v)
var = sum((z - media) ** 2 for z in v) / len(v)
print('varianza =', var, '→ división por cero')
try:
    print([(z - media) / math.sqrt(var) for z in v])
except ZeroDivisionError as exc:
    print('ZeroDivisionError:', exc)

## 12. Corrección


In [ ]:
print('con epsilon:', [round(z, 6) for z in layer_norm([2.0, 2.0, 2.0])])
print('→ el epsilon no es un detalle de implementación: evita un NaN que se propaga a toda la red')

## 13. Desafío guiado

Cuenta los parámetros de la FFN (d_model=512, d_ff=2048) y compáralos con los de la atención multi-cabeza del mismo bloque.


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **el andamiaje sin el que la atención no entrena**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

Con el bloque completo se ensamblan encoder y decoder, y aparecen los límites (T08).
